In [ ]:
import init

In [ ]:
epochs = 20

In [ ]:
import tensorflow as tf


def get_dataset(x, y, batch_size=128, shuffle_buffer=10_000, drop_remainder=True):
    """Scale image bytes to diffusion space and batch aligned labels.

    The affine transform does not clip values. No prefetching, repetition, validation
    split, or deterministic shuffle seed is added. Labels remain aligned with images
    while shuffling.

    Args:
        x (np.ndarray): Image values nominally in [0, 255]. Trailing image dimensions
            are preserved; supply the channel dimension required by the model.
        y (np.ndarray): Labels with the same leading length as x. Values and label shape
            are preserved.
        batch_size (int): Number of examples in each dataset batch. Defaults to ``128``.
        shuffle_buffer (int | None): None preserves input order; a positive integer
            enables shuffling without an explicit operation seed. TensorFlow global
            seeding can still control this randomness. Defaults to ``10000``.
        drop_remainder (bool): True discards an incomplete final batch; False retains
            it. Defaults to ``True``.

    Returns:
        tf.data.Dataset: Finite (images, labels) batches, with float32 image values
        transformed as 2*x/255-1 and the configured remainder policy.
    """

    x = x.astype("float32") / 255.0
    x = (x * 2.0) - 1.0
    # x = x[..., None]

    dataset = tf.data.Dataset.from_tensor_slices((x, y))

    # A supplied buffer enables random shuffling; None keeps input order.
    if shuffle_buffer is not None:
        dataset = dataset.shuffle(shuffle_buffer)

    dataset = dataset.batch(batch_size, drop_remainder=drop_remainder)

    return dataset

In [ ]:
from tensorflow.keras import datasets, layers

import numpy as np


(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()

x_train = np.expand_dims(x_train, -1)
x_train = layers.ZeroPadding2D((2, 2))(x_train).numpy()

x_test = np.expand_dims(x_test, -1)
x_test = layers.ZeroPadding2D((2, 2))(x_test).numpy()

trainset = get_dataset(x_train, y_train)
valset = get_dataset(x_test, y_test, shuffle_buffer=None, drop_remainder=False)

In [ ]:
from diffusion.models.wrapper.diffusion_classifier_v2 import DiffusionClassifierV2
from diffusion.models.transformer.di_t_classifier import DiTClassifier


vit = DiTClassifier(
    image_size=32, 
    dim_forced=False, 
    cond_type="labels", 
    depth=15, 
    connection_ids_dict={8: (3,), 10: (1,), 12: (7,)}, 
    cross_attention_ids_dict={13: (9,), 15: (11,)}, 
    cross_attention_kwargs={
        "use_layer_norm": True, 
        "ln_no_adaptation": False
    }, 
    vit_block_ids=[1, 3, 5, 13, 15], 
    use_decoder_ids=[13, 15], 
    vit_block_mlp_output_dims={1: 32, 3: 64, 5: 64, 13: 32, 15: 32}, 
    downsample_ids=[2, 4], 
    downsample_kwargs={
        "pos_embed_type": "2d_sincos", 
    }, 
    upsample_ids=[12, 14], 
    upsample_kwargs={
        "pos_embed_type": "2d_sincos", 
        "scaling_method": "interpolate", 
        "scaling_interpolation_method": "bilinear"
    }, 
    reshaper_ids_dict={
        6: "flatten", 7: "unflatten", 
        8: "flatten", 9: "unflatten", 
        10: "flatten", 11: "unflatten"
    }, 
    reshaper_kwargs={
        "add_kl": True, 
        "latent_dim_ratio": [1 / 64, 1 / 256, 1 / 512]
    }, 
    final_ffn_activation_func="tanh", 
    aggregate_from_noises=True, 
)
model = DiffusionClassifierV2(
    network=vit, 
    p_uncond=0., 
    test_cfg_scale=None, 
    swap_noise_image=True, 
    kl_loss_coef=0.01, 
    modify_first_t=True, 
    train_noisified_max_timesteps=0, 
    test_noisified_max_timesteps=0, 
) # 

vit.summary()

In [ ]:
from tensorflow.keras import optimizers


lr_schedule = optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3, 
    decay_steps=epochs * len(trainset), #  * 11
)

model.compile(
    optimizer=optimizers.Adam(lr_schedule), 
    loss="mse", 
    # run_eagerly=True
)

In [ ]:
from tensorflow.keras import callbacks

from diffusion import ImageGenerator
from common.callbacks.lr_logger import LrLogger


callbacks_list = [
    LrLogger(), 
    callbacks.ProgbarLogger(count_mode="steps"), 
    ImageGenerator(), 
]

In [ ]:
history = model.fit_generator(
    x=trainset, 
    epochs=epochs, 
    validation_data=valset, 
    callbacks=callbacks_list, 
).history

In [ ]:
history = model.fit_discriminator(
    x=trainset, 
    epochs=epochs, 
    validation_data=valset, 
    callbacks=callbacks_list[:-1], 
).history

In [ ]:
# history = model.fit_progressively(
#     x=trainset.take(1), 
#     num_stages=10, 
#     stage_epochs=epochs, 
#     pacing_type="plateau", 
#     monitor="val_noise_loss", 
#     min_delta=1e-3, 
#     patience=1, 
#     validation_data=valset, 
#     callbacks=callbacks_list, 
# ).history

In [ ]:
model.evaluate(x=valset, eval_both=True, network_name="ema")

In [ ]:
model.evaluate(x=valset, eval_both=True, network_name="raw")

In [ ]:
from common.utils import plot_images


imgs = model.sample(add_null_label=True, scale=3., steps=1_000, eta=1., verbose=True)
plot_images(imgs, has_null_label=True)

In [ ]:
imgs = model.sample(add_null_label=True, scale=None, steps=1_000, verbose=True)
plot_images(imgs, has_null_label=True)

In [ ]:
imgs = model.sample(add_null_label=True, verbose=True)
plot_images(imgs, has_null_label=True)

In [ ]:
from common.utils import plot_history


plot_history(history)

In [ ]:
plot_history(history, show_all_x_ticks=False, range_=(10, None))